# 🏇 TJK Ganyan Bot — Training Pipeline V2

**Sıra:**
1. Repo clone + dependencies
2. TJK'dan historical sonuç scrape (son 6 ay)
3. 82 feature'lı races.csv üret
4. Feature audit
5. Retrain V2 (AGF bağımlılığını kıran)
6. Backtest
7. Yeni model dosyalarını indir

⚠️ TJK scraping: ~180 gün × ~10 hipodrom = ~30 dk (checkpoint'lu, kesilirse devam eder)


## 1. Setup — Repo Clone + Dependencies

In [ ]:
# Repo clone
!rm -rf tjk-ganyan-bot
!git clone https://github.com/BerkayMangal/tjk-ganyan-bot.git
%cd tjk-ganyan-bot

# Dependencies
!pip install -q requests beautifulsoup4 pandas numpy scikit-learn xgboost lightgbm catboost joblib tqdm

print("✅ Setup tamamlandı")

## 2. Historical Data Scraper — Fonksiyonlar

In [ ]:
import requests
import re
import csv
import io
import json
import time
import logging
import os
from datetime import date, datetime, timedelta
from typing import Optional, List, Dict
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('tjk_history')

TJK_BASE = "https://www.tjk.org"
TJK_CDN = "https://medya-cdn.tjk.org/raporftp/TJKPDF"

SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept-Language': 'tr-TR,tr;q=0.9',
})

HIPODROMLAR = {
    'İstanbul': 59, 'Ankara': 6, 'İzmir': 35, 'Bursa': 16, 'Adana': 1,
    'Elazığ': 23, 'Kocaeli': 41, 'Diyarbakır': 21, 'Şanlıurfa': 63, 'Antalya': 7,
}

CDN_NAME_MAP = {
    'İstanbul': 'Istanbul', 'İzmir': 'Izmir', 'Şanlıurfa': 'Sanliurfa',
    'Elazığ': 'Elazig', 'Diyarbakır': 'Diyarbakir',
}


def _try_csv_url(url):
    try:
        resp = SESSION.get(url, timeout=15)
        if resp.status_code != 200:
            return None
        for enc in ['utf-8', 'windows-1254', 'iso-8859-9', 'latin-1']:
            try:
                text = resp.content.decode(enc)
                break
            except (UnicodeDecodeError, LookupError):
                continue
        else:
            return None
        for delim in [';', ',', '\t']:
            try:
                reader = csv.reader(io.StringIO(text), delimiter=delim)
                rows = list(reader)
                if len(rows) > 2 and len(rows[0]) > 5:
                    return rows
            except:
                continue
    except Exception:
        pass
    return None


def _try_results_csv(target_date, sehir_name):
    yyyy = target_date.strftime('%Y')
    yyyy_mm_dd = target_date.strftime('%Y-%m-%d')
    dd_mm_yyyy = target_date.strftime('%d.%m.%Y')
    url_name = CDN_NAME_MAP.get(sehir_name, sehir_name)
    patterns = [
        f"{TJK_CDN}/{yyyy}/{yyyy_mm_dd}/CSV/GunlukYarisSonuclari/{dd_mm_yyyy}-{url_name}-GunlukYarisSonuclari-TR.csv",
        f"{TJK_CDN}/{yyyy}/{yyyy_mm_dd}/CSV/GunlukYarisSonucu/{dd_mm_yyyy}-{url_name}-GunlukYarisSonucu-TR.csv",
    ]
    for url in patterns:
        rows = _try_csv_url(url)
        if rows:
            return rows
    return None


def _try_program_csv(target_date, sehir_name):
    yyyy = target_date.strftime('%Y')
    yyyy_mm_dd = target_date.strftime('%Y-%m-%d')
    dd_mm_yyyy = target_date.strftime('%d.%m.%Y')
    url_name = CDN_NAME_MAP.get(sehir_name, sehir_name)
    url = f"{TJK_CDN}/{yyyy}/{yyyy_mm_dd}/CSV/GunlukYarisProgrami/{dd_mm_yyyy}-{url_name}-GunlukYarisProgrami-TR.csv"
    return _try_csv_url(url)


def _parse_csv_rows(rows):
    header = rows[0]
    hl = [h.strip().lower() for h in header]
    def find(*names):
        for n in names:
            for i, h in enumerate(hl):
                if n in h:
                    return i
        return None

    ci = {
        'race': find('koşu no', 'kosu no', 'koşu', 'race'),
        'finish': find('sıra', 'derece', 'finish', 'sonuç'),
        'number': find('at no', 'no', 'numara'),
        'name': find('at adı', 'at adi', 'isim', 'name'),
        'age': find('yaş', 'yas', 'age'),
        'sire': find('baba', 'sire'),
        'dam': find('anne', 'dam'),
        'dam_sire': find('anne babası', 'anababa', 'damsire'),
        'weight': find('kilo', 'sıklet', 'weight'),
        'jockey': find('jokey', 'jockey'),
        'trainer': find('antrenör', 'antrenor', 'trainer'),
        'form': find('son 6', 'form'),
        'kgs': find('kgs'),
        'hp': find('hp', 'handikap', 'handicap'),
        'start': find('st', 'start'),
        'distance': find('mesafe', 'distance'),
        'track': find('pist', 'track'),
        'ganyan': find('ganyan', 'odds'),
        'prize': find('ikramiye', 'prize', 'ödül'),
    }

    def get(row, key, default=''):
        idx = ci.get(key)
        if idx is not None and idx < len(row):
            return row[idx].strip()
        return default

    races = {}
    for row in rows[1:]:
        if len(row) < 3:
            continue
        rn_s = get(row, 'race')
        rm = re.search(r'\d+', rn_s)
        race_num = int(rm.group()) if rm else 0

        num_s = get(row, 'number', '0')
        nm = re.search(r'\d+', num_s)
        if not nm:
            continue
        horse_number = int(nm.group())
        horse_name = get(row, 'name')
        if not horse_name:
            continue

        fp_s = get(row, 'finish', '0')
        fp_m = re.search(r'(\d+)', fp_s)
        finish_pos = int(fp_m.group(1)) if fp_m else 0

        wt = get(row, 'weight', '57')
        wm = re.search(r'[\d]+[,.]?\d*', wt)
        weight = float(wm.group().replace(',', '.')) if wm else 57.0

        at = get(row, 'age', '4')
        am = re.search(r'(\d)', at)
        age = int(am.group(1)) if am else 4

        kgs_s = get(row, 'kgs', '0')
        km = re.search(r'\d+', kgs_s)
        kgs = int(km.group()) if km else 0

        hp_s = get(row, 'hp', '0')
        hm = re.search(r'\d+', hp_s)
        hp = int(hm.group()) if hm else 0

        gn = get(row, 'ganyan', '0')
        gm = re.search(r'[\d]+[,.]?\d*', gn)
        ganyan = float(gm.group().replace(',', '.')) if gm else 0

        horse = {
            'horse_number': horse_number, 'horse_name': horse_name,
            'finish_position': finish_pos, 'age': age, 'age_text': get(row, 'age'),
            'weight': weight, 'jockey_name': get(row, 'jockey'),
            'trainer_name': get(row, 'trainer'), 'sire': get(row, 'sire'),
            'dam': get(row, 'dam'), 'dam_sire': get(row, 'dam_sire'),
            'form': get(row, 'form'), 'equipment': '', 'kgs': kgs,
            'handicap': hp, 'start_position': horse_number, 'ganyan': ganyan,
        }

        if race_num not in races:
            dist_s = get(row, 'distance', '0')
            dm = re.search(r'\d+', dist_s)
            dist = int(dm.group()) if dm else 0
            races[race_num] = {
                'race_number': race_num, 'distance': dist,
                'track_type': get(row, 'track'), 'prize': 0, 'horses': [],
            }
        races[race_num]['horses'].append(horse)
    return races


def scrape_day(target_date):
    all_horses = []
    for sehir_name, sehir_id in HIPODROMLAR.items():
        races = {}

        # Sonuç CSV dene
        rows = _try_results_csv(target_date, sehir_name)
        if rows:
            races = _parse_csv_rows(rows)

        # Program CSV fallback
        if not races:
            rows = _try_program_csv(target_date, sehir_name)
            if rows:
                races = _parse_csv_rows(rows)

        if not races:
            continue

        for rnum, race in sorted(races.items()):
            horses = race.get('horses', [])
            has_results = any(h.get('finish_position', 0) > 0 for h in horses)
            if len(horses) < 3:
                continue

            race_id = f"{target_date.strftime('%Y%m%d')}_{sehir_name}_{rnum}"
            for h in horses:
                h['race_id'] = race_id
                h['race_number'] = rnum
                h['race_date'] = target_date.strftime('%Y-%m-%d')
                h['hippodrome'] = sehir_name
                h['distance'] = race.get('distance', 0)
                h['track_type'] = race.get('track_type', '')
                h['first_prize'] = race.get('prize', 0)
                h['n_runners'] = len(horses)
                h['has_results'] = has_results

            all_horses.extend(horses)

        time.sleep(0.3)  # hipodrom arası bekleme

    return all_horses

print("✅ Scraper fonksiyonları hazır")

## 2b. Hızlı Test — Tek gün dene

In [ ]:
# Önce tek gün test et — TJK CSV endpoint çalışıyor mu?
from datetime import date, timedelta

test_date = date.today() - timedelta(days=3)  # 3 gün öncesi
print(f"Test tarihi: {test_date}")

test_rows = scrape_day(test_date)
print(f"Sonuç: {len(test_rows)} kayıt")

if test_rows:
    df_test = pd.DataFrame(test_rows)
    print(f"Hipodromlar: {df_test['hippodrome'].unique().tolist()}")
    print(f"Yarış sayısı: {df_test['race_id'].nunique()}")
    has_fp = (df_test['finish_position'] > 0).sum()
    print(f"Finish position olan: {has_fp}/{len(df_test)}")
    print()
    print("İlk 3 kayıt:")
    print(df_test[['race_id','horse_name','finish_position','jockey_name','weight']].head(3).to_string())
else:
    print("⚠️ Veri gelmedi — TJK erişimi kontrol et")

## 2c. Full Scrape — Son 180 gün

⚠️ **~30 dakika sürer.** Checkpoint var — runtime kesilirse tekrar çalıştırınca kaldığı yerden devam eder.

`DAYS_BACK` değerini değiştirerek kısaltabilirsin (90 gün de yeterli başlangıç).


In [ ]:
# 📌 AYARLAR
DAYS_BACK = 180       # Son kaç gün
RATE_LIMIT = 0.8      # Saniye/gün (her gün 10 hipodrom deneniyor, arası 0.3s)
OUTPUT_DIR = 'data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Checkpoint kontrolü
checkpoint_path = os.path.join(OUTPUT_DIR, 'scrape_checkpoint.json')
if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        checkpoint = json.load(f)
    resume_date = datetime.strptime(checkpoint['last_date'], '%Y-%m-%d').date()
    total_records = checkpoint.get('total_records', 0)
    print(f"♻️ Checkpoint: {checkpoint['last_date']} — {total_records:,} kayıt — devam")
else:
    resume_date = None
    total_records = 0

today = date.today()
start = today - timedelta(days=DAYS_BACK)
if resume_date and resume_date >= start:
    start = resume_date + timedelta(days=1)

n_days = (today - start).days
print(f"📅 {start} → {today} ({n_days} gün)")
print(f"⏱️ Tahmini: ~{n_days * 0.8 / 60:.0f} dakika")

In [ ]:
# 🚀 SCRAPE
csv_path = os.path.join(OUTPUT_DIR, 'races_raw.csv')
csv_exists = os.path.exists(csv_path) and total_records > 0

csv_file = open(csv_path, 'a' if csv_exists else 'w', newline='', encoding='utf-8')
writer = None
day_count = 0
empty_days = 0

current = start
pbar = tqdm(total=(today - start).days, desc="🏇 Scraping TJK")

while current <= today:
    try:
        rows = scrape_day(current)

        if rows:
            if writer is None:
                fieldnames = sorted(rows[0].keys())
                writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
                if not csv_exists:
                    writer.writeheader()
                    csv_exists = True

            for row in rows:
                writer.writerow(row)

            total_records += len(rows)
            n_results = sum(1 for r in rows if r.get('finish_position', 0) > 0)
            pbar.set_postfix(date=str(current), rows=len(rows), res=n_results, total=total_records)
        else:
            empty_days += 1

        day_count += 1

        # Checkpoint her 10 günde
        if day_count % 10 == 0:
            csv_file.flush()
            with open(checkpoint_path, 'w') as f:
                json.dump({'last_date': str(current), 'total_records': total_records}, f)

    except Exception as e:
        if day_count < 5:
            print(f"⚠️ {current}: {e}")

    current += timedelta(days=1)
    pbar.update(1)
    time.sleep(RATE_LIMIT)

pbar.close()
csv_file.close()

# Final checkpoint
with open(checkpoint_path, 'w') as f:
    json.dump({'last_date': str(today), 'total_records': total_records}, f)

print(f"\n✅ Scrape tamamlandı!")
print(f"   {day_count} gün tarandı | {empty_days} boş gün")
print(f"   {total_records:,} toplam kayıt → {csv_path}")

## 2d. Veri Kontrolü

In [ ]:
df_raw = pd.read_csv(os.path.join(OUTPUT_DIR, 'races_raw.csv'), low_memory=False)
print(f"Raw data: {len(df_raw):,} kayıt")
print(f"Yarış: {df_raw['race_id'].nunique()}")
print(f"Hipodrom: {df_raw['hippodrome'].nunique()} — {df_raw['hippodrome'].value_counts().to_dict()}")
print(f"Tarih aralığı: {df_raw['race_date'].min()} → {df_raw['race_date'].max()}")

if 'finish_position' in df_raw.columns:
    has_fp = (df_raw['finish_position'] > 0).sum()
    print(f"Sonuçlu kayıt: {has_fp:,} / {len(df_raw):,} ({has_fp/len(df_raw):.0%})")

# Eksik kolonlar
for col in ['jockey_name', 'trainer_name', 'sire', 'dam', 'form', 'weight']:
    filled = df_raw[col].notna().sum() if col in df_raw.columns else 0
    pct = filled / len(df_raw) * 100
    print(f"  {col:15s}: {pct:.0f}% dolu")

## 3. Feature Engineering — 82 Feature CSV

In [ ]:
import sys
sys.path.insert(0, '.')
from model.features import FeatureBuilder

# Sadece sonuçlu yarışlar
df_results = df_raw[
    df_raw['finish_position'].notna() & (df_raw['finish_position'] > 0)
].copy()
print(f"Sonuçlu kayıt: {len(df_results):,}")

# Feature builder
fb = FeatureBuilder()
fb_ok = fb.load()
print(f"FeatureBuilder: {'✅' if fb_ok else '❌'} — {len(fb.feature_cols)} feature")

In [ ]:
# Her yarış için feature hesapla
feature_rows = []
skipped = 0

for race_id, group in tqdm(df_results.groupby('race_id'), desc="⚙️ Feature building"):
    try:
        horses = group.to_dict('records')
        n = len(horses)
        if n < 2:
            skipped += 1
            continue

        row0 = horses[0]
        race_info = {
            'distance': row0.get('distance', 1400),
            'track_type': row0.get('track_type', 'kum'),
            'group_name': '',
            'hippodrome_name': row0.get('hippodrome', ''),
            'first_prize': row0.get('first_prize', 100000),
            'temperature': 15, 'humidity': 60,
            'race_date': row0.get('race_date'),
        }

        # AGF proxy: ganyan'dan
        agf_data = []
        for h in horses:
            ganyan = h.get('ganyan', 0)
            pct = min(100.0 / ganyan, 80) if ganyan and ganyan > 0 else 100.0 / n
            agf_data.append({'horse_number': h.get('horse_number', 0), 'agf_pct': pct, 'is_ekuri': False})
        total_pct = sum(a['agf_pct'] for a in agf_data)
        if total_pct > 0:
            for a in agf_data:
                a['agf_pct'] = a['agf_pct'] / total_pct * 100

        fb_horses = [{
            'horse_number': h.get('horse_number', 0),
            'horse_name': h.get('horse_name', ''),
            'jockey_name': h.get('jockey_name', ''),
            'trainer_name': h.get('trainer_name', ''),
            'weight': h.get('weight', 57),
            'age': h.get('age', 4),
            'age_text': h.get('age_text', ''),
            'form': h.get('form', ''),
            'equipment': h.get('equipment', ''),
            'kgs': h.get('kgs', 0),
            'last_20_score': 0,
            'handicap': h.get('handicap', 0),
            'gate_number': h.get('start_position', h.get('horse_number', 1)),
            'sire': h.get('sire', h.get('sire_name', '')),
            'dam': h.get('dam', h.get('dam_name', '')),
            'dam_sire': h.get('dam_sire', h.get('dam_sire_name', '')),
            'total_earnings': h.get('total_earnings', 0),
        } for h in horses]

        matrix, names = fb.build_race_features(fb_horses, race_info, agf_data)

        for i, h in enumerate(horses):
            frow = {
                'race_id': race_id,
                'race_date': h.get('race_date'),
                'hippodrome': h.get('hippodrome', ''),
                'horse_name': h.get('horse_name', ''),
                'horse_number': h.get('horse_number', 0),
                'finish_position': h.get('finish_position', 0),
                'n_runners': n,
            }
            for j, col in enumerate(fb.feature_cols):
                frow[col] = float(matrix[i, j])
            feature_rows.append(frow)

    except Exception as e:
        skipped += 1

print(f"\n✅ {len(feature_rows):,} kayıt | {skipped} skip")

In [ ]:
# CSV kaydet
df_feat = pd.DataFrame(feature_rows)
FEAT_CSV = os.path.join(OUTPUT_DIR, 'races_featured.csv')
df_feat.to_csv(FEAT_CSV, index=False)
print(f"✅ {FEAT_CSV} — {len(df_feat):,} kayıt")

# Fill rate kontrolü
fcols = [c for c in df_feat.columns if c.startswith('f_')]
fills = {c: (df_feat[c] != 0).mean() for c in fcols}
avg = np.mean(list(fills.values()))
dead = sum(1 for v in fills.values() if v < 0.01)
print(f"Ortalama fill: {avg:.0%} | Dead features: {dead}/{len(fcols)}")

# En kötü 5
for name, rate in sorted(fills.items(), key=lambda x: x[1])[:5]:
    print(f"  {name:30s} {rate:.1%}")

## 4. Feature Audit

In [ ]:
!python train/feature_audit.py --data {FEAT_CSV} --model-dir model/trained

## 5. Retrain V2

AGF bağımlılığını kıran, 2-pass training, ablated model dahil.


In [ ]:
!python train/retrain_v2.py \
  --data {FEAT_CSV} \
  --output model/trained \
  --labels exponential \
  --agf-noise 0.05 \
  --test-ratio 0.20

## 6. Backtest

In [ ]:
!python train/backtester.py --data {FEAT_CSV} --model-dir model/trained

## 7. Sonuçlar

In [ ]:
# Retrain sonuçları
meta_path = 'model/trained/train_meta_v2.json'
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)

    print("=" * 60)
    print("RETRAIN V2 SONUÇLARI")
    print("=" * 60)
    print(f"Label: {meta.get('label_method')} | AGF noise: {meta.get('agf_noise')}")
    print(f"Train: {meta.get('train_records', 0):,} | Test: {meta.get('test_records', 0):,}")
    print()

    for name, ev in meta.get('eval', {}).items():
        if ev:
            print(f"  {name:12s} | Top1={ev.get('top1_accuracy', 0):.1%} | "
                  f"Top3={ev.get('top3_accuracy', 0):.1%} | "
                  f"NDCG@1={ev.get('ndcg1', 0):.3f}")

    print()
    for name, imp in meta.get('agf_importance', {}).items():
        if imp is not None:
            s = "🔴" if imp > 0.5 else "🟡" if imp > 0.35 else "✅"
            print(f"  AGF ({name}): {imp:.1%} {s}")

# Backtest
bt_path = 'model/trained/backtest_report.json'
if os.path.exists(bt_path):
    with open(bt_path) as f:
        bt = json.load(f)
    print()
    print("=" * 60)
    print("BACKTEST")
    print("=" * 60)
    print(f"DAR ayak:    {bt.get('dar_leg_hit_rate', 0):.1%}")
    print(f"GENİŞ ayak:  {bt.get('genis_leg_hit_rate', 0):.1%}")
    print(f"DAR kupon:   {bt.get('dar_kupon_hit_rate', 0):.1%}")
    print(f"GENİŞ kupon: {bt.get('genis_kupon_hit_rate', 0):.1%}")
    print(f"Top-1:       {bt.get('winner_top1_rate', 0):.1%}")
    print(f"Top-3:       {bt.get('winner_top3_rate', 0):.1%}")

## 8. Model İndir + Railway Push

In [ ]:
import zipfile
from google.colab import files

model_files = [
    'model/trained/xgb_ranker.pkl', 'model/trained/lgbm_ranker.pkl',
    'model/trained/cb_ranker.pkl', 'model/trained/ablated_ranker.pkl',
    'model/trained/ablated_columns.json', 'model/trained/scaler.pkl',
    'model/trained/feature_columns.json', 'model/trained/rstats.json',
    'model/trained/train_meta_v2.json', 'model/trained/backtest_report.json',
]

zip_path = 'data/trained_model_v2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in model_files:
        if os.path.exists(fp):
            zf.write(fp, os.path.basename(fp))
            print(f"  ✅ {os.path.basename(fp)}")
        else:
            print(f"  ⚠️ {os.path.basename(fp)} — yok")

print(f"\n📦 {zip_path}")
files.download(zip_path)

In [ ]:
# Opsiyonel: Direkt GitHub push
# !git config user.email "you@example.com"
# !git config user.name "BerkayMangal"
# !git add model/trained/
# !git commit -m "retrain v2: agf-breaking, 2-pass, ablated model"
# !git push https://<GITHUB_TOKEN>@github.com/BerkayMangal/tjk-ganyan-bot.git main